In [ ]:
# General libraries
import os
import re
import random
import pickle
import statistics
import collections
from collections import Counter
from itertools import combinations

# Data handling
import numpy as np
import pandas as pd

# Visualization
import seaborn as sns
from matplotlib import pyplot as plt, cm, colors, colorbar
from matplotlib_venn import venn2
from mpl_toolkits.axes_grid1 import make_axes_locatable
from adjustText import adjust_text

# Machine learning & preprocessing
from sklearn.linear_model import LinearRegression, RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.manifold import TSNE
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score, pairwise_distances
)

# Dimensionality reduction
import umap
import umap.umap_ as umap_module  # if you need the lower-level API

# Feature selection
from boruta import BorutaPy

# Shapelet learning
from pyts.classification import LearningShapelets
from pyts.datasets import load_gunpoint
from pyts.utils import windowed_view

# Statistical tests and models
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statannot import add_stat_annotation
#import scikit_posthocs as sp
import shap
#from xgboost import XGBRegressor
from scipy import stats
from scipy.stats import (
    ttest_ind, levene, mannwhitneyu, shapiro, mstats,
    pearsonr, kruskal, skew
)
from scipy.spatial import ConvexHull, convex_hull_plot_2d

# Progress bar
from tqdm import tqdm
from statsmodels.formula.api import ols
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore') 
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, ttest_ind, f_oneway
from statsmodels.stats.multitest import multipletests
import warnings
from scipy.stats import chi2_contingency
from collections import Counter
# ----------------------------------------------------------------------
# Example initializations (optional)
scaler = StandardScaler()
mmscaler = MinMaxScaler()
pca = PCA(n_components=2, svd_solver='full')

# Classifiers
rdg = RidgeClassifier(alpha=0.5)
mlp = MLPClassifier(random_state=1, max_iter=300, activation='relu')
lgr = LogisticRegression(random_state=1, max_iter=500)
DT = DecisionTreeClassifier(random_state=0, max_depth=10)
adb = AdaBoostClassifier(n_estimators=100, random_state=0)
gbc = GradientBoostingClassifier(n_estimators=100, random_state=1)
knn = KNeighborsClassifier(n_neighbors=3)
SGD = SGDClassifier(loss='log', random_state=1, max_iter=100, early_stopping=True,
                    learning_rate='optimal', validation_fraction=0.2)
rf = RandomForestClassifier(max_depth=10, random_state=0)
clf_svm = SVC(kernel='rbf')


In [ ]:
def extract_connectivity(band,data):
    Y=[]
    coh_ar=np.zeros([len(data),88*88])
    for i in range(0,len(data)):
        m=np.loadtxt(data[i])[band*88:(band+1)*88,:] # extract only delta band 
        m=np.tril(m, k=-1).flatten()  ## Take upper/lower Triangle of the Symetrical Coherence Matrix
        coh_ar[i,:]=m
    coh_ar_zscored = stats.zscore(coh_ar, axis=0) # Within Subject Z-Transform
    return coh_ar_zscored

org_directory="/home/jupy/Data_SourceFC/T0"
ls_org_directory=os.listdir(org_directory)
directory_T0=[item for item in ls_org_directory if item !='.ipynb_checkpoints' ]
directory_T0_ob=[x for x in directory_T0 if x.startswith("F")]
directory_T0_lean=[x for x in directory_T0 if x.startswith("L")]

org_directory="/home/jupy/Data_SourceFC/T45"
ls_org_directory=os.listdir(org_directory)
directory_T45=[item for item in ls_org_directory if item != '.ipynb_checkpoints']
directory_T45_ob=[x for x in directory_T45 if x.startswith("F")]
directory_T45_lean=[x for x in directory_T45 if x.startswith("L")]

In [ ]:
def extract_key(filename):
    return filename[0:4]
    
def align_subjects_btw_T0_T45 (directory_list_T0, directory_list_T45):
    # Create dictionaries for all time points
    time_points = {
        'T0': {extract_key(f): f for f in directory_list_T0 if extract_key(f)},
        'T45': {extract_key(f): f for f in directory_list_T45 if extract_key(f)}}    
    # Find common keys across ALL time points
    common_keys = set(time_points['T0'])  # Start with T0 keys
    for tp in time_points:
        common_keys &= set(time_points[tp])  # Intersect with each time point
        # Extract aligned files for each time point (sorted by key)
    aligned_files = {
        tp: [time_points[tp][key] for key in sorted(common_keys)]
        for tp in time_points }
    # Find unaligned files for each time point
    unique_files = {
        tp: [f for key, f in time_points[tp].items() if key not in common_keys]
        for tp in time_points}
    
    excl0=unique_files['T0']
    aligned_directory_T0=[item for item in directory_list_T0 if item not in excl0 ]
    excl45=unique_files['T45']
    aligned_directory_T45=[item for item in directory_list_T45 if item not in excl45]
    print ('Length T0: ',len(aligned_directory_T0),'    Length T45',len(aligned_directory_T45)) 
    
    # Create dictionaries with 4-digit keys
    dict_T0 = {extract_key(f): f for f in aligned_directory_T0 if extract_key(f)}
    dict_T45 = {extract_key(f): f for f in aligned_directory_T45 if extract_key(f)}
    # Find aligned keys
    common_keys = set(dict_T0) & set(dict_T45)
    aligned_T0 = [dict_T0[key] for key in common_keys]
    aligned_T45 = [dict_T45[key] for key in common_keys]
    # Find unaligned elements
    unique_T0 = [f for key, f in dict_T0.items() if key not in common_keys]
    unique_T45 = [f for key, f in dict_T45.items() if key not in common_keys]
    # Sort both aligned lists by key (first 4 digits)
    aligned_pairs = sorted(zip(aligned_T0, aligned_T45), key=lambda x: extract_key(x[0]))
    aligned_T0_sorted, aligned_T45_sorted = zip(*aligned_pairs) if aligned_pairs else ([], [])
    # Final sorted lists (now aligned by first 4 digits)
    finaldir_T0 = list(aligned_T0_sorted) 
    finaldir_T45 = list(aligned_T45_sorted) 
    
    return finaldir_T0, finaldir_T45


In [ ]:
finaldir_T0_lean,finaldir_T45_lean=align_subjects_btw_T0_T45 (directory_T0_lean, directory_T45_lean)
finaldir_T0_ob,finaldir_T45_ob=align_subjects_btw_T0_T45 (directory_T0_ob, directory_T45_ob)

In [ ]:
################  Use If Analyse Each Individual Band
def get_fc_data_per_T(fc_data_dir, subject_list_dir, band="alpha"):
    os.chdir(fc_data_dir)
    band_map = {
        "delta": 0,
        "theta": 1,
        "alpha": 2,
        "beta": 3,
        "gamma": 4 }  
    
    if band not in band_map:
        raise ValueError(f"Invalid band '{band}'. Choose from {list(band_map.keys())}.")  
    fc_data = extract_connectivity(band_map[band], subject_list_dir)
    return fc_data

#### Within Band Per Subject Z-transform
Study to back up **within band standardization** (Z-transform)
<br>EEG Frequency Bands in Psychiatric Disorders: A Review of Resting State Studies (https://www.frontiersin.org/journals/human-neuroscience/articles/10.3389/fnhum.2018.00521/full)
<br>*"We emphasize the need to use a standardized definition for each frequency band, based on the most commonly used non-overlapping frequencies: (delta: <4 Hz; theta: 4–7.5 Hz; alpha: 7.5–12.5 Hz; beta: 12.5–30 Hz; gamma: 30–40 Hz)."


In [ ]:
band='gamma'
bands=[band]

In [ ]:
fc_T0_ob=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T0",finaldir_T0_ob,band)
fc_T45_ob=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T45",finaldir_T45_ob,band)
fc_T0_lean=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T0",finaldir_T0_lean,band)
fc_T45_lean=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T45",finaldir_T45_lean,band)

### ----------------- Extract FC with ROI Names ------------------

In [ ]:
# Define ROIs and frequency bands
rois = [f'roi{i+1}' for i in range(88)]  
print(bands)

column_names = []
for band in bands:
    for source_roi in rois:
        for target_roi in rois:
            column_names.append(f"{source_roi}_{target_roi}_{band}")

# Verify we have the right number of features: 5 bands * 88 source ROIs * 88 target ROIs 
print(f"Generated {len(column_names)} column names")

fc_T45_ob= pd.DataFrame(fc_T45_ob, columns=column_names).abs()
fc_T0_ob= pd.DataFrame(fc_T0_ob, columns=column_names).abs()
fc_T45_lean= pd.DataFrame(fc_T45_lean, columns=column_names).abs()
fc_T0_lean= pd.DataFrame(fc_T0_lean, columns=column_names).abs()

def extract_roi_columns_regex(fc_df, roi_numbers): #Extract columns where column names contain any of the specified ROI numbers
    # Remove 0 from the list since your columns start from roi1
    roi_numbers = [roi for roi in roi_numbers if roi != 0]    
    # Create regex pattern to match roiX where X is any of the numbers
    pattern = r'roi(' + '|'.join(map(str, roi_numbers)) + r')(_|$)'
    # Filter columns that match the pattern
    matching_columns = [
        col for col in fc_df.columns 
        if re.search(pattern, col) ]
    return fc_df[matching_columns]

#acc_roi_numbers = [1, 20, 21, 42, 43, 54, 55, 56, 57, 86, 88]
#acc_roi_numbers=[85, 41, 53, 42, 87, 19, 20, 54, 55, 56]
acc_roi_numbers=np.arange(1,89).tolist()

acc_fc_T45_ob = extract_roi_columns_regex(fc_T45_ob, acc_roi_numbers)
acc_fc_T0_ob = extract_roi_columns_regex(fc_T0_ob, acc_roi_numbers)
acc_fc_T45_lean = extract_roi_columns_regex(fc_T45_lean, acc_roi_numbers)
acc_fc_T0_lean = extract_roi_columns_regex(fc_T0_lean, acc_roi_numbers)

compute **weighted degree connectivity** (i.e. strength) 

In [ ]:
def calculate_specific_roi_connectivity_sum_efficient(acc_fcdf, roi_list):
    # Convert ROI numbers to the format used in column names
    roi_strings = [f"roi{roi}" for roi in roi_list]
    # Create empty result dataframe
    result_df = pd.DataFrame(0, index=acc_fcdf.index, columns=roi_strings)
    # Pre-process column information
    column_info = []
    for col in acc_fcdf.columns:
        parts = col.split('_')
        roi1, roi2 = parts[0], parts[1]
        column_info.append((roi1, roi2, col))
    # For each target ROI, sum all connections involving it
    for roi in roi_strings:
        # Find all columns where this ROI appears
        roi_columns = []
        for roi1, roi2, col_name in column_info:
            if roi1 == roi or roi2 == roi: ########### Self-connection excluded , Symmetrical pair only cont one of the unique #################
                roi_columns.append(col_name) 
        # Sum all connections involving this ROI
        if roi_columns:
            result_df[roi] = acc_fcdf[roi_columns].sum(axis=1)  
    return result_df

In [ ]:
roi_connectivity_T0_ob = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T0_ob, acc_roi_numbers)
roi_connectivity_T45_ob = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T45_ob, acc_roi_numbers)
roi_connectivity_T0_lean = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T0_lean, acc_roi_numbers)
roi_connectivity_T45_lean = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T45_lean, acc_roi_numbers)

from scipy.stats import zscore

roi_connectivity_T0_ob = (roi_connectivity_T0_ob).apply(zscore)
roi_connectivity_T45_ob = (roi_connectivity_T45_ob ).apply(zscore)
roi_connectivity_T0_lean = (roi_connectivity_T0_lean ).apply(zscore)
roi_connectivity_T45_lean =(roi_connectivity_T45_lean).apply(zscore)

### ANCOVA 

`model.resid`: These are the residuals – the part of the T45 value that the model could not explain using T0. They represent the "adjusted T45" values after removing the linear influence of T0.
<br>`model.params['Intercept']`: Addthe intercept back to "re-center" the residuals around the grand mean of T45. Without this, the residuals would have a mean of zero. Adding the intercept back creates values that are on the original scale, which is much more intuitive for plotting and further analysis (e.g., group comparisons).

In [ ]:
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan


connections = roi_connectivity_T0_lean.columns
groups = {'Lean': (roi_connectivity_T0_lean, roi_connectivity_T45_lean),
          'Obese': (roi_connectivity_T0_ob, roi_connectivity_T45_ob)}

# DataFrames to store assumption check results
assumption_results = pd.DataFrame(index=connections, 
                                  columns=pd.MultiIndex.from_product([['Lean', 'Obese'], 
                                                                     ['Linear_pval', 'Normality_pval', 'Homoscedasticity_pval', 'Slope']]))

# DataFrames to store the FINAL adjusted T45 values
t45_adjusted_lean = pd.DataFrame(index=roi_connectivity_T45_lean.index)
t45_adjusted_obese = pd.DataFrame(index=roi_connectivity_T45_ob.index)

In [ ]:
# Function to check assumptions and adjust data
def check_assumptions_and_adjust(t0_data, t45_data, group_name, conn):
    """
    Checks assumptions for a single connection within a single group.
    Returns adjusted T45 values and assumption results.
    """
    df = pd.DataFrame({'T0': t0_data, 'T45': t45_data}).dropna()
    if len(df) < 3:  # Need at least 3 points for regression
        return np.nan, {'Linear_pval': np.nan, 'Normality_pval': np.nan, 
                       'Homoscedasticity_pval': np.nan, 'Slope': np.nan}
    
    # Fit the model
    model = ols('T45 ~ T0', data=df).fit()
    
    # Get predictions and residuals
    predicted = model.fittedvalues
    residuals = model.resid
    
    # 1. Check Linearity (Using Rainbow test)
    try:
        # The rainbow test checks if the linear fit is adequate
        rainbow_stat, rainbow_pval = sm.stats.diagnostic.linear_rainbow(model)
    except:
        rainbow_pval = np.nan
    
    # 2. Check Normality of Residuals (Shapiro-Wilk test)
    if len(residuals) > 3:  # Shapiro-Wilk requires 3+ samples
        _, normality_pval = stats.shapiro(residuals)
    else:
        normality_pval = np.nan
    
    # 3. Check Homoscedasticity (Breusch-Pagan test)
    try:
        # The Breusch-Pagan test checks if variance of residuals is constant
        bp_lm, bp_pval, _, _ = het_breuschpagan(residuals, model.model.exog)
    except:
        bp_pval = np.nan
    
    # Calculate adjusted T45 values
    adjusted_values = residuals + model.params['Intercept']
    
    results = {
        'Linear_pval': rainbow_pval,
        'Normality_pval': normality_pval,
        'Homoscedasticity_pval': bp_pval,
        'Slope': model.params['T0']
    }
    
    return adjusted_values, results

In [ ]:
for idx, (group_name, (t0_df, t45_df)) in enumerate(groups.items()):
    print(f"\nProcessing {group_name} group...")
    
    for conn in connections:
        # Get data for this connection and group
        t0_data = t0_df[conn]
        t45_data = t45_df[conn]
        
        # Check assumptions and get adjusted values
        adjusted_values, results = check_assumptions_and_adjust(t0_data, t45_data, group_name, conn)
        
        # Store results
        for key, value in results.items():
            assumption_results.loc[conn, (group_name, key)] = value
        
        # Store adjusted values
        if group_name == 'Lean':
            t45_adjusted_lean[conn] = adjusted_values
        else:
            t45_adjusted_obese[conn] = adjusted_values
        
# --------------------------------------------------------------------
# SUMMARIZE ASSUMPTION VIOLATIONS
# --------------------------------------------------------------------
print("\n" + "="*60)
print("ASSUMPTION CHECK SUMMARY")
print("="*60)

alpha = 0.05
for group in ['Lean', 'Obese']:
    print(f"\n--- {group} Group ---")
    linear_violations = (assumption_results[(group, 'Linear_pval')] < alpha).sum()
    normality_violations = (assumption_results[(group, 'Normality_pval')] < alpha).sum()
    homosced_violations = (assumption_results[(group, 'Homoscedasticity_pval')] < alpha).sum()
    
    print(f"Connections violating Linearity (Rainbow test p < {alpha}): {linear_violations}/{len(connections)}")
    print(f"Connections violating Normality (Shapiro-Wilk p < {alpha}): {normality_violations}/{len(connections)}")
    print(f"Connections violating Homoscedasticity (Breusch-Pagan p < {alpha}): {homosced_violations}/{len(connections)}")

In [ ]:
# --------------------------------------------------------------------
# ADDRESS VIOLATIONS - ROBUST ADJUSTMENT
# --------------------------------------------------------------------
print("\n" + "="*60)
print("ADDRESSING ASSUMPTION VIOLATIONS")
print("="*60)

# Create new DataFrames for robustly adjusted values
t45_adjusted_robust_lean = pd.DataFrame(index=roi_connectivity_T45_lean.index)
t45_adjusted_robust_obese = pd.DataFrame(index=roi_connectivity_T45_ob.index)

# Apply robust methods for connections with violations
for conn in connections:
    for group_name, (t0_df, t45_df) in groups.items():
        t0_data = t0_df[conn]
        t45_data = t45_df[conn]
        df = pd.DataFrame({'T0': t0_data, 'T45': t45_data}).dropna()
        
        if len(df) < 3:
            continue
            
        # Check if this connection has severe violations
        linear_pval = assumption_results.loc[conn, (group_name, 'Linear_pval')]
        norm_pval = assumption_results.loc[conn, (group_name, 'Normality_pval')]
        hets_pval = assumption_results.loc[conn, (group_name, 'Homoscedasticity_pval')]
        
        # If severe linearity violation, use non-parametric adjustment
        if pd.notna(linear_pval) and linear_pval < 0.01:
            # Use rank-based transformation
            df['T0_rank'] = df['T0'].rank()
            df['T45_rank'] = df['T45'].rank()
            model = ols('T45_rank ~ T0_rank', data=df).fit()
            adjusted_ranks = model.resid + model.params['Intercept']
            # Convert back to original scale using quantiles
            adjusted_values = adjusted_ranks.rank(pct=True).apply(lambda x: np.quantile(df['T45'], x))
            
        # If severe non-normality or heteroscedasticity, use robust regression
        elif (pd.notna(norm_pval) and norm_pval < 0.01) or (pd.notna(hets_pval) and hets_pval < 0.01):
            # Use Huber robust regression
            huber_t = sm.RLM(df['T45'], sm.add_constant(df['T0']), M=sm.robust.norms.HuberT())
            huber_results = huber_t.fit()
            predicted = huber_results.predict(sm.add_constant(df['T0']))
            residuals = df['T45'] - predicted
            adjusted_values = residuals + huber_results.params[0]  # Add intercept
            
        else:
            # Use standard OLS adjustment
            model = ols('T45 ~ T0', data=df).fit()
            adjusted_values = model.resid + model.params['Intercept']
        
        # Store robustly adjusted values
        if group_name == 'Lean':
            t45_adjusted_robust_lean[conn] = adjusted_values
        else:
            t45_adjusted_robust_obese[conn] = adjusted_values

# --------------------------------------------------------------------
# SAVE THE RESULTS
# --------------------------------------------------------------------
print (f'### Current Band: {band} ###')
# Save assumption results
#assumption_results.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_Within_Group_Assumption_Check_Results.csv')
#t45_adjusted_robust_lean.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_T45_Adjusted_Robust_Lean.csv')
#t45_adjusted_robust_obese.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_T45_Adjusted_Robust_Obese.csv')

print(f"\nFinal adjusted datasets ready for within-group analysis.")
print(f"Lean group shape: {t45_adjusted_robust_lean.shape}")
print(f"Obese group shape: {t45_adjusted_robust_obese.shape}")

adjusted_fc_ob=t45_adjusted_robust_obese 
adjusted_fc_lean=t45_adjusted_robust_lean

### Umap

In [ ]:
opt_n_neighbors_ls=[15]
opt_min_dist_ls=[0.1]
opt_k_ls=[2]
bands_ls=np.array(['gamma'])

In [ ]:
from umap.umap_ import UMAP as UMAP

def visualize_lean_obese_umap(roi_connectivity_T0_ob, roi_connectivity_T0_lean, n_neighbors, min_dist):
    # Prepare the data
    ob_data = np.array(roi_connectivity_T0_ob)
    lean_data = np.array(roi_connectivity_T0_lean)
    
    ob_data_scaled=ob_data
    lean_data_scaled=lean_data
    
    # Independent UMAP for each group
    # For obese samples: independent UMAP
    umap_ob = UMAP(n_components=2, random_state=42, n_neighbors=n_neighbors, min_dist=min_dist)
    ob_embedding = umap_ob.fit_transform(ob_data_scaled)
    
    # For lean samples: independent UMAP
    umap_lean = UMAP(n_components=2, random_state=42, n_neighbors=n_neighbors, min_dist=min_dist)
    lean_embedding = umap_lean.fit_transform(lean_data_scaled)
    
    # Create the visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot 1: Independent UMAP for obese group
    axes[0].scatter(ob_embedding[:, 0], ob_embedding[:, 1], 
                   c='red', label='Obese', alpha=0.7, s=30)
    axes[0].set_title('Independent UMAP: Obese Group')
    axes[0].set_xlabel('UMAP 1')
    axes[0].set_ylabel('UMAP 2')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Independent UMAP for lean group
    axes[1].scatter(lean_embedding[:, 0], lean_embedding[:, 1], 
                   c='blue', label='Lean', alpha=0.7, s=30)
    axes[1].set_title('Independent UMAP: Lean Group')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Return only the independent embeddings
    results_dict = {
        'ob_embedding': ob_embedding,
        'lean_embedding': lean_embedding
    }
    
    return fig, results_dict, umap_ob

In [ ]:
n_neighbors, min_dist=opt_n_neighbors_ls[ np.where(bands_ls==band)[0][0]], opt_min_dist_ls[ np.where(bands_ls==band)[0][0]]
umap_fig, umap_results,umap_model_ob = visualize_lean_obese_umap( adjusted_fc_ob, adjusted_fc_lean, n_neighbors, min_dist)

### Use shap value to find the roi contribute to Umap-D1 seperation, 

In [ ]:
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

def analyze_umap_shap(umap_results, hormone_data_ob, group_name='45_adj_0'):
    # Get UMAP embeddings for obese group only
    umap_embedding = umap_results['ob_embedding']

    results = {}
    for component in [0, 1]:
        print(f"\n=== Analyzing UMAP Component {component + 1} for Obese Group ({group_name}) ===")
        
        # Target is the UMAP coordinate
        y = umap_embedding[:, component]
        X = hormone_data_ob.values
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42)
        
        # Train a model to predict UMAP coordinates
        model = XGBRegressor(random_state=42)
        model.fit(X_train, y_train)
        
        # Evaluate model
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        print(f"R² score for UMAP component {component + 1}: {r2:.3f}")
        
        # Calculate SHAP values
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        
        # Create summary plot with reduced figure size
        plt.figure(figsize=(5, 4))  # Reduced from (6, 5)
        shap.summary_plot(shap_values, X_test, 
                         feature_names=hormone_data_ob.columns,
                         show=False)
        plt.title(f'SHAP Summary - Obese UMAP {component + 1} ({group_name})\nR² = {r2:.3f}')
        plt.tight_layout()
        plt.show()
        
        # Get mean absolute SHAP values for feature importance
        mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
        feature_importance = pd.DataFrame({
            'feature': hormone_data_ob.columns,
            'mean_abs_shap': mean_abs_shap
        }).sort_values('mean_abs_shap', ascending=False)
        
        print(f"\nTop features for UMAP component {component + 1}:")
        print(feature_importance.head(10))
        
        # Store results
        results[f'component_{component}'] = {
            'model': model,
            'shap_values': shap_values,
            'feature_importance': feature_importance,
            'r2_score': r2
        }
    
    return results

In [ ]:
shap_results=analyze_umap_shap(umap_results,  adjusted_fc_ob, group_name='45_adj_0')

In [ ]:
os.chdir('/home/jupy/Subtypes_Obesity_Clustering/')
# Save to file
with open(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/ShapValues/{band}_shap.pkl', 'wb') as f:
    pickle.dump(shap_results, f)

# Load from file
#with open(f'shap_results{band}.pkl', 'rb') as f:
 #   loaded_dict = pickle.load(f)

##########################################################################################


In [ ]:
def cluster_global_umap(results_dict, n_clusters=3, embedding_group='ob_embedding'):
    global_embedding = results_dict[embedding_group]
    
    if embedding_group == 'ob_embedding':
        group_labels = np.array([1] * global_embedding.shape[0])
        group_name = 'Obese'
        group_color = 'red'
    else:
        group_labels = np.array([0] * global_embedding.shape[0])
        group_name = 'Lean'
        group_color = 'blue'
    
    # K-means clustering with flexible number of clusters on global UMAP
    kmeans_global = KMeans(n_clusters=n_clusters, random_state=42)
    global_clusters = kmeans_global.fit_predict(global_embedding)
    
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    
    # Define markers for clusters - using a list of common markers
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', 'X', 'd']
    
    # Plot Global UMAP with k-means clusters - Only the specified group
    group_mask = group_labels == (1 if embedding_group == 'ob_embedding' else 0)
    group_embedding_subset = global_embedding[group_mask]
    group_clusters_subset = global_clusters[group_mask]
    
    for cluster_id in range(n_clusters):
        cluster_mask = group_clusters_subset == cluster_id
        if np.any(cluster_mask):
            ax.scatter(group_embedding_subset[cluster_mask, 0], 
                      group_embedding_subset[cluster_mask, 1],
                      c=group_color, marker=markers[cluster_id % len(markers)],
                      label=f'{group_name} - Cluster {cluster_id}',
                      alpha=0.7, s=50)
    
    #ax.set_title(f'Global UMAP: {group_name} K-means Clustering ({n_clusters} clusters)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Adjust legend to prevent overcrowding
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Prepare cluster results dictionary
    cluster_results = {
        'global_clusters': global_clusters,
        'kmeans_global': kmeans_global
    }
    
    return fig, cluster_results

In [ ]:
n_clusters=opt_k_ls[ np.where(bands_ls==band)[0][0]]
fig_cluster_obese, cluster_results_obese = cluster_global_umap(umap_results, n_clusters=n_clusters, embedding_group='ob_embedding')
#fig_cluster_obese.savefig(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs_Updated/{band}_cluster_obese.pdf", dpi=300)

In [ ]:
umap_df = pd.DataFrame(umap_results['ob_embedding'], columns=['UMAP1', 'UMAP2'])
#umap_df.to_csv(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/R_inputs/umap_embeddings_{band}.csv', index=False)

cluster_df = pd.DataFrame({
    'cluster': cluster_results_obese['global_clusters']})
#cluster_df.to_csv(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/R_inputs/cluster_labels_{band}.csv', index_label='sample_id')

### Test distances between clusters are statistically significant (in R) 

gamma - yes
delta - yes
theta - yes
alpha - yes
beta - yes

### Check if Obesity-specific

<br> First compute obese and lean samples **distances to cluster centers**
<br>Then used **mann-whiteney (One-sided)** to test if **lean distances are greater** than obese distances, p<0.05 means lean participants are significantly farther from obese cluster centers.
<br>**Chi-sqare test** the **cluster distribution** of lean and obese group, to see if obese and lean group fall into the same general clusters (p<0.05) means cluster distribution are significantly differnt

In [ ]:
import joblib
from sklearn.metrics.pairwise import pairwise_distances

umap_model_obese = umap_model_ob
kmeans_model_obese = cluster_results_obese['kmeans_global']
lean_umap_projected = umap_model_obese.transform(adjusted_fc_lean)

# Get cluster centers from the obese-trained k-means
obese_centers = kmeans_model_obese.cluster_centers_
# Calculate distances from each lean subject to each obese cluster center
distances = pairwise_distances(lean_umap_projected, obese_centers)
# Assign each lean subject to the closest obese cluster
lean_cluster_assignments = np.argmin(distances, axis=1)

# Calculate average distance to assigned cluster center
lean_distances_to_center = np.min(distances, axis=1)
obese_distances_to_center = kmeans_model_obese.transform(umap_results['ob_embedding'])
obese_distances_to_center = np.min(obese_distances_to_center, axis=1)

print(f"Mean distance to cluster center:")
print(f"Obese group: {np.mean(obese_distances_to_center):.3f}")
print(f"Lean group: {np.mean(lean_distances_to_center):.3f}")

# Mann-Whitney U test (one-sided: lean > obese)
u_stat, p_value = stats.mannwhitneyu(
    lean_distances_to_center, 
    obese_distances_to_center, 
    alternative='greater'  # tests if lean > obese
)

print(f"Mann-Whitney U test (lean > obese): U={u_stat:.3f}, p={p_value:.3f}") 

# Check if lean subjects are concentrated in one cluster
from collections import Counter
lean_cluster_counts = Counter(lean_cluster_assignments)
obese_cluster_counts = Counter(kmeans_model_obese.labels_)

print("\nCluster distribution:")
print("Obese:", dict(obese_cluster_counts))
print("Lean:", dict(lean_cluster_counts))

# Chi-square test for different distributions
from scipy.stats import chi2_contingency
# Create contingency table
contingency_table = np.zeros((2, len(np.unique(cluster_results_obese['global_clusters']))))
for i in range(contingency_table.shape[1]):
    contingency_table[0, i] = obese_cluster_counts.get(i, 0)
    contingency_table[1, i] = lean_cluster_counts.get(i, 0)
    
chi2, chi_p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-square test for different distributions: p={chi_p:.3f}")

# gamma:
Mean distance to cluster center:
Obese group: 0.872
Lean group: 1.817
Mann-Whitney U test (lean > obese): U=980.000, p=0.000

Cluster distribution:
Obese: {0: 12, 1: 18}
Lean: {0: 16, 1: 17}
Chi-square test for different distributions: p=0.672


# Clincial Test 

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

#### Read Demographical,Anthropometrical,Clincial, BE

In [ ]:
def process_demo_data (demo_df):
    sorted_demo_data = demo_df.replace('#NULL!', np.nan)
    for col in sorted_demo_data.columns:
        sorted_demo_data[col] = pd.to_numeric(sorted_demo_data[col], errors='ignore')
    sorted_demo_data = sorted_demo_data.fillna(sorted_demo_data.mean(numeric_only=True))
    return sorted_demo_data
    
demo_data=pd.read_csv('/home/jupy/Subtypes_Obesity_Clustering/DemographicalData_BL.csv')
demo_data=demo_data.drop(columns=['ADD_SLAIENCE2','ADD_EUPHORIA2','ADD_TOLERANCE2','ADD_WITHDRAWAL2',
 'ADD_CONFLIC12','ADD_CONFLICT22', 'ADD_RELAPSE2','ADD_IDENTIFICATION2'])

demodf_lean=demo_data[demo_data['scr_no'].isin( [f[0:3] for f in finaldir_T0_lean] )] # f[0:3] becuase lean people ID e.g. L03 is 3 digit
demodf_ob=demo_data[demo_data['scr_no'].isin( [f[0:4] for f in finaldir_T0_ob] )] # f[0:4] becuase obese people ID e.g. F246 is 4 digit
print([f[0:4] for f in finaldir_T0_ob]==[f[0:4] for f in finaldir_T45_ob]) # if True means T0 and T45 have the same ID of subject  

sorted_demo_data_ob=process_demo_data (demodf_ob).reset_index(drop=True)
sorted_demo_data_lean=process_demo_data (demodf_lean).reset_index(drop=True)

noal_sorted_demo_data_lean=sorted_demo_data_lean
noal_sorted_demo_data_ob=sorted_demo_data_ob

scr_list = noal_sorted_demo_data_lean['scr_no'].tolist()
noal_finaldir_T0_lean=[f for f in finaldir_T0_lean if any(scr in f for scr in scr_list)] 
noal_finaldir_T45_lean=[f for f in finaldir_T45_lean if any(scr in f for scr in scr_list)] 

scr_list = noal_sorted_demo_data_ob['scr_no'].tolist()
noal_finaldir_T0_ob = [f for f in finaldir_T0_ob if any(scr in f for scr in scr_list)] 
noal_finaldir_T45_ob = [f for f in finaldir_T45_ob if any(scr in f for scr in scr_list)] 

In [ ]:
from scipy.stats import zscore
noal_sorted_demo_data_ob=noal_sorted_demo_data_ob.drop(columns=['scr_no','Category','Reproductive_status'])

for i in [ 'BL_SATIS15', 'ADD_CHOC' , 'ADD_EUPHORIA' ]:
    noal_sorted_demo_data_ob[i]=noal_sorted_demo_data_ob[i].round().astype(int)
noal_sorted_demo_data_ob=noal_sorted_demo_data_ob.drop(columns=['IE_TOTAL'])

In [ ]:
#### Add cluster label

In [ ]:
unique_clusters = np.sort(np.unique(cluster_results_obese['global_clusters']))
cluster_dfs = []
cluster_labels = []

for cluster_id in unique_clusters:
    cluster_data = noal_sorted_demo_data_ob[cluster_results_obese['global_clusters'] == cluster_id]
    cluster_dfs.append(cluster_data)
    cluster_labels.extend([cluster_id] * cluster_data.shape[0])
noal_sorted_demo_data_ob = pd.concat(cluster_dfs, axis=0).reset_index(drop=True)
noal_sorted_demo_data_ob['clusterID'] = cluster_labels

In [ ]:
noal_sorted_demo_data_ob.head(2)

In [ ]:
cat_anthro_vars=['DRD5','Taq1a','OPRNM1', 'OPRM_Addiction','FTO','smoking'] # Chi-square test
ordi_anthro_vars=[ 'alcohol', 'Drink'] # Mann-Whitney U test or Kruskal-Wallis H test depending on how many groups to test
num_vars_anthro=['age', 'bmi','Insulin_Resistance'] # depending on how many groups to test: T-test (if normal) or Mann-Whitney U (if non-normal) if 2 groups,
# otherwise ANOVA (if normal) or Kruskal-Wallis H (if non-normal)

vars_glucose=['glucB_0', 'glucB_45'] 
vars_insulin=['insB_0', 'insB_45']
vars_ghrelin=['GhrelinB_0', 'GhrelinB_45']
vars_glp1=['GLP1B_0', 'GLP1B_45']
vars_pyy=[  'PYYB_0', 'PYYB_45']

#### Standardize Numerical data # Z_transform

In [ ]:
for i in num_vars_anthro:
    noal_sorted_demo_data_ob[i]= (noal_sorted_demo_data_ob[i] - noal_sorted_demo_data_ob[i].mean()) / noal_sorted_demo_data_ob[i].std()
noal_sorted_demo_data_ob.head(3)

#### Test Anthropometric Varibales

In [ ]:
def check_var_normality(data, group_col, var_col, alpha=0.05):
    """Check normality for each group using Shapiro-Wilk test"""
    groups = data[group_col].unique()
    all_normal = True
    
    for group in groups:
        group_data = data[data[group_col] == group][var_col].dropna()
        if len(group_data) < 3:  # Shapiro-Wilk requires at least 3 observations
            return False
        _, p_value = stats.shapiro(group_data)
        if p_value < alpha:
            all_normal = False
            break
    return all_normal

def perform_statistical_tests_anthro(df, cluster_col='clusterID'):
    """Perform statistical tests for different variable types and return results"""    
    results = []
    
    # Get number of unique clusters
    n_clusters = df[cluster_col].nunique()
    
    # Categorical variables - Chi-square test
    for var in cat_anthro_vars:
        if var in df.columns:
            # Create contingency table
            contingency_table = pd.crosstab(df[var], df[cluster_col])
            
            # Check if expected frequencies are sufficient (all >= 5)
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            
            # If expected frequencies are too low, use Fisher's exact test
            if np.any(expected < 5):
                try:
                    from scipy.stats import fisher_exact
                    # For 2x2 tables
                    if contingency_table.shape == (2, 2):
                        _, p_value = fisher_exact(contingency_table)
                    else:
                        # For larger tables, use Monte Carlo simulation
                        chi2, p_value, dof, expected = chi2_contingency(
                            contingency_table, simulate_p_value=True, replicates=1000
                        )
                    test_used = "Fisher's exact (Monte Carlo)"
                except:
                    test_used = "Chi-square (with low expected frequencies)"
            else:
                test_used = "Chi-square"
            
            results.append({
                'Variable': var,
                'Type': 'Categorical',
                'Test': test_used,
                'Statistic': chi2,
                'P_value': p_value,
                'Groups': n_clusters
            })
    
    # Ordinal variables - Mann-Whitney U or Kruskal-Wallis
    for var in ordi_anthro_vars:
        if var in df.columns:
            groups = [df[df[cluster_col] == cluster][var].dropna().values 
                     for cluster in df[cluster_col].unique()]
            
            # Remove groups with no data
            groups = [g for g in groups if len(g) > 0]
            
            if len(groups) < 2:
                continue
            if n_clusters == 2:
                # Mann-Whitney U test for 2 groups
                stat, p_value = mannwhitneyu(groups[0], groups[1])
                test_used = "Mann-Whitney U"
            else:
                # Kruskal-Wallis H test for >2 groups
                stat, p_value = kruskal(*groups)
                test_used = "Kruskal-Wallis H"
            
            results.append({
                'Variable': var,
                'Type': 'Ordinal',
                'Test': test_used,
                'Statistic': stat,
                'P_value': p_value,
                'Groups': n_clusters
            })
    
    # Numerical variables - T-test/ANOVA or Mann-Whitney U/Kruskal-Wallis
    for var in num_vars_anthro:
        if var in df.columns:
            groups = [df[df[cluster_col] == cluster][var].dropna().values 
                     for cluster in df[cluster_col].unique()]
            
            # Remove groups with no data
            groups = [g for g in groups if len(g) > 0]
            
            if len(groups) < 2:
                continue
            
            # Check normality
            is_normal = check_var_normality(df, cluster_col, var)
            
            if n_clusters == 2:
                if is_normal:
                    # T-test for 2 groups
                    stat, p_value = ttest_ind(groups[0], groups[1], equal_var=False)
                    test_used = "T-test (Welch's)"
                else:
                    # Mann-Whitney U test for 2 groups
                    stat, p_value = mannwhitneyu(groups[0], groups[1])
                    test_used = "Mann-Whitney U"
            else:
                if is_normal:
                    # ANOVA for >2 groups
                    stat, p_value = f_oneway(*groups)
                    test_used = "ANOVA"
                else:
                    # Kruskal-Wallis H test for >2 groups
                    stat, p_value = kruskal(*groups)
                    test_used = "Kruskal-Wallis H"
            
            results.append({
                'Variable': var,
                'Type': 'Numerical',
                'Test': test_used,
                'Statistic': stat,
                'P_value': p_value,
                'Groups': n_clusters
            })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Apply FDR correction using Benjamini-Hochberg method
    if not results_df.empty:
        p_values = results_df['P_value'].values
        rejected, pvals_corrected, _, _ = multipletests(
            p_values, alpha=0.05, method='fdr_bh'
        )
        
        results_df['P_value_FDR'] = pvals_corrected
        results_df['Significant_FDR'] = rejected
    
    return results_df


In [ ]:
anthro_results = perform_statistical_tests_anthro(noal_sorted_demo_data_ob, 'clusterID')
anthro_results 

In [ ]:
# Save results to CSV
anthro_results.to_csv(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/Anthron_Test/{band}_anthro.csv', index=False)
#print(f"\nResults saved to 'statistical_test_results.csv'")

#### Test Hormone Vars

##### Z transform

In [ ]:
for i in ['glucB', 'insB', 'GhrelinB', 'GLP1B', 'PYYB']:
    cc=noal_sorted_demo_data_ob[i+'_0']
    cc= (cc - cc.mean()) / cc.std()
    cc = cc - cc.min().min() + 1  # ensures all values > 0
    noal_sorted_demo_data_ob[i+'_0_z']=cc
    
    zz=noal_sorted_demo_data_ob[i+'_45']
    zz= (zz - zz.mean()) / zz.std()
    zz = zz - zz.min().min() + 1
    noal_sorted_demo_data_ob[i+'_45_z']=zz

#### Permutation ANCOVA
log transform following Z-transform

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import r2_score
from scipy import stats
from statsmodels.stats.anova import anova_lm

def permutation_ancova(data, dep_var, covar, group_var, n_permutations=1000):
    """
    Perform permutation ANCOVA
    Returns: original F-statistic, p-value, and effect sizes
    """
    # Original model
    formula = f'{dep_var} ~ {covar} + C({group_var})'
    original_model = ols(formula, data=data).fit()
    
    # Get original F-statistic for group effect
    # We need to extract the F-value for the group effect from ANOVA table
    anova_table = anova_lm(original_model, typ=2)
    original_f = anova_table.loc[f'C({group_var})', 'F']
    
    # Permutation test
    perm_f_stats = []
    
    for i in range(n_permutations):
        # Permute the group labels while keeping covariate-outcome relationship
        perm_data = data.copy()
        perm_data[group_var] = resample(data[group_var], replace=False, random_state=i)
        
        # Fit model with permuted groups
        try:
            perm_model = ols(formula, data=perm_data).fit()
            perm_anova = anova_lm(perm_model, typ=2)
            perm_f = perm_anova.loc[f'C({group_var})', 'F']
            perm_f_stats.append(perm_f)
        except:
            # If model fails (e.g., singular matrix), skip this permutation
            continue
    
    # Calculate p-value
    perm_f_stats = np.array(perm_f_stats)
    p_value = np.mean(perm_f_stats >= original_f)
    
    # Calculate effect sizes
    partial_eta_squared = calculate_partial_eta2(original_model, group_var)
    
    return {
        'original_f': original_f,
        'p_value': p_value,
        'partial_eta_squared': partial_eta_squared,
        'r_squared': original_model.rsquared,
        'adj_r_squared': original_model.rsquared_adj,
        'n_permutations': len(perm_f_stats),
        'model': original_model
    }

def calculate_partial_eta2(model, group_var):
    """Calculate partial eta squared for group effect"""
    anova_table = anova_lm(model, typ=2)
    ss_group = anova_table.loc[f'C({group_var})', 'sum_sq']
    ss_residual = anova_table.loc['Residual', 'sum_sq']
    return ss_group / (ss_group + ss_residual)


In [ ]:
# Save all model summaries to one text file
with open(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/ANCOVA/{band}_ancova_FDRcorrected.txt', 'w') as f:
    # Main analysis with permutation ANCOVA
    for horm in ['insB', 'GhrelinB', 'GLP1B', 'glucB', 'PYYB']:
        f.write(f"\n{'='*60}\n")
        f.write(f"PERMUTATION ANCOVA (1000 permutations): {horm}\n")
        f.write(f"{'='*60}\n")
        
        # Perform permutation ANCOVA
        datadf=noal_sorted_demo_data_ob.copy().sort_values('clusterID').reset_index(drop=True)
        logdf = datadf.copy()
        logdf.loc[:, datadf.columns != 'clusterID'] = np.log(datadf.loc[:, datadf.columns != 'clusterID'])
        perm_results = permutation_ancova ( #simple_permutation_ancova(
            data= logdf ,
            dep_var=   f'{horm}_45_z' ,
            covar= f'{horm}_0_z',
            group_var='clusterID',
            n_permutations=1000
        )
        
        f.write("PERMUTATION RESULTS:\n")
        #f.write(f"R² difference (group effect): {perm_results['original_r2_diff']:.4f}\n")
        f.write(f"Permutation p-value: {perm_results['p_value']:.4f}\n")
        f.write(f"Number of successful permutations: {perm_results['n_permutations']}\n")
        
        f.write("\nMODEL FIT:\n")
        f.write(f"Full model R²: {perm_results['r_squared']:.4f}\n")
        f.write(f"Adjusted model R²: {perm_results['adj_r_squared']:.4f}\n")
        #f.write(f"Full model R²: {perm_results['full_r2']:.4f}\n")
        #f.write(f"Reduced model R²: {perm_results['reduced_r2']:.4f}\n")
        f.write(f"Partial η² (group effect): {perm_results['partial_eta_squared']:.4f}\n")
        
        # Extract p-values for cluster comparisons and apply FDR correction
        #model = perm_results['full_model']
        model = perm_results['model']
        cluster_params = [param for param in model.params.index if 'clusterID' in param]
        cluster_pvalues = [model.pvalues[param] for param in cluster_params]
        
        # Apply FDR correction to the 3 cluster comparisons
        from statsmodels.stats.multitest import multipletests
        fdr_corrected = multipletests(cluster_pvalues, alpha=0.05, method='fdr_bh')
        
        f.write("\nPARAMETER ESTIMATES (from full model) - FDR CORRECTED:\n")
        for i, param in enumerate(cluster_params):
            value = model.params[param]
            se = model.bse[param]
            t = model.tvalues[param]
            raw_p = model.pvalues[param]
            fdr_p = fdr_corrected[1][i]
            
            # Use FDR-corrected p-values for significance stars
            stars = "***" if fdr_p < 0.001 else "**" if fdr_p < 0.01 else "*" if fdr_p < 0.05 else ""
            f.write(f"  {param:30}: {value:8.4f} ± {se:.4f} (t = {t:.3f}, raw p = {raw_p:.4f}, FDR p = {fdr_p:.4f}) {stars}\n")
        
        # Write non-cluster parameters without FDR correction
        non_cluster_params = [param for param in model.params.index if 'clusterID' not in param]
        for param in non_cluster_params:
            value = model.params[param]
            se = model.bse[param]
            t = model.tvalues[param]
            p = model.pvalues[param]
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            f.write(f"  {param:30}: {value:8.4f} ± {se:.4f} (t = {t:.3f}, p = {p:.4f}) {stars}\n")
        
        # Interpret permutation p-value (overall model effect)
        if perm_results['p_value'] < 0.001:
            significance = "***"
        elif perm_results['p_value'] < 0.01:
            significance = "**"
        elif perm_results['p_value'] < 0.05:
            significance = "*"
        else:
            significance = "ns"
        
        f.write(f"\nPermutation test result: p = {perm_results['p_value']:.4f} {significance}\n")
        f.write(f"FDR correction applied to {len(cluster_params)} cluster comparisons\n")
        
        # Save the full model summary
        f.write(f"\n{'='*80}\n")
        f.write(f"FULL MODEL SUMMARY FOR {horm}:\n")
        f.write(f"{'='*80}\n")
        #f.write(str(perm_results['full_model'].summary()))
        f.write(str(perm_results['model'].summary()))
        f.write(f"\n\n{'='*80}\n\n")

print("All permutation ANCOVA results with FDR-corrected cluster comparisons saved!")

#### Test Behavior Vars

In [ ]:
def check_var_normality(data, group_col, var_col, alpha=0.05):
    """Check normality for each group using Shapiro-Wilk test"""
    groups = data[group_col].unique()
    all_normal = True
    
    for group in groups:
        group_data = data[data[group_col] == group][var_col].dropna()
        if len(group_data) < 3:  # Shapiro-Wilk requires at least 3 observations
            return False
        _, p_value = stats.shapiro(group_data)
        if p_value < alpha:
            all_normal = False
            break
    return all_normal

def perform_statistical_tests(df,ordi_behav_vars ,cluster_col='clusterID'):
    """Perform statistical tests for different variable types and return results"""    
    results = []
    
    # Get number of unique clusters
    n_clusters = df[cluster_col].nunique()
    
    # Ordinal variables - Mann-Whitney U or Kruskal-Wallis
    for var in ordi_behav_vars:
        if var in df.columns:
            groups = [df[df[cluster_col] == cluster][var].dropna().values 
                     for cluster in df[cluster_col].unique()]
            
            # Remove groups with no data
            groups = [g for g in groups if len(g) > 0]
            
            if len(groups) < 2:
                continue
                
            if n_clusters == 2:
                # Mann-Whitney U test for 2 groups
                stat, p_value = mannwhitneyu(groups[0], groups[1])
                test_used = "Mann-Whitney U"
            else:
                # Kruskal-Wallis H test for >2 groups
                stat, p_value = kruskal(*groups)
                test_used = "Kruskal-Wallis H"
            
            results.append({
                'Variable': var,
                'Type': 'Ordinal',
                'Test': test_used,
                'Statistic': stat,
                'P_value': p_value,
                'Groups': n_clusters
            })

    results_df = pd.DataFrame(results)
    
    # Apply FDR correction using Benjamini-Hochberg method
    if not results_df.empty:
        p_values = results_df['P_value'].values
        rejected, pvals_corrected, _, _ = multipletests(
            p_values, alpha=0.05, method='fdr_bh'
        )
        
        results_df['P_value_FDR'] = pvals_corrected
        results_df['Significant_FDR'] = rejected
    
    return results_df

vars_behav_IE=['IE_UPE', 'IE_EPR', 'IE_RHSC', 'IE_BFCC', 'IE_TOTAL']
vars_behav_DEB=['DEB_R','DEB_EX', 'DEB_EM']
vars_behav_YFAS=['YFAS_SYMP','YFAS_DICH']
vars_behav_BISBAS=['BISBAS_BIS', 'BISBAS_BASDRIVE', 'BISBAS_BASFUN', 'BISBAS_REWARD']
vars_behav_HPS=['HPS_HM', 'HPS_GF', 'HPS_CF', 'HPS_NAF']
vars_behav_addict=['ADD_CHOC', 'ADD_EX', 'ADD_ALC', 'ADD_CIG', 'ADD_REC', 'ADD_CAF',
       'ADD_GAMB', 'ADD_MUS', 'ADD_INT', 'ADD_SHOP', 'ADD_WORK',
       'ADD_LOVE',  'ADD_SALIENCE', 'ADD_EUPHORIA',
       'ADD_TOLERANCE', 'ADD_WITHDRAWAL', 'ADD_CONFLICT1',
       'ADD_CONFLICT2', 'ADD_RELAPSE', 'ADD_IDENTIFICATION']
vars_NRS_enjoy= ['BL_DRtaste1','BL_DRenjoy1','BL_DRtaste2', 'BL_DRenjoy2']
vars_NRS_hunger=['BL_HUNGER0',  'BL_HUNGER45']
vars_NRS_satisf=['BL_SATIS0', 'BL_SATIS45' ]
vars_NRS_full=['BL_FULL0', 'BL_FULL45']
vars__NRS_howmuch=['BL_HOWMUCH0', 'BL_HOWMUCH45']
vars_NRS_liketo=['BL_LIKETO0', 'BL_LIKETO45']
vars_behave_MIND= ['MIND']
vars_behave_BINGE= ['BINGE']

In [ ]:
behav_results=[]

for ordi_behav_vars in [vars_behav_IE, vars_behav_DEB,vars_behav_YFAS, vars_behav_BISBAS,
    vars_behav_HPS, vars_behav_addict, vars_NRS_enjoy,  vars_NRS_hunger, vars_NRS_full,
    vars__NRS_howmuch, vars_NRS_liketo, vars_behave_MIND, vars_behave_BINGE ]:    
    behav_df=perform_statistical_tests(noal_sorted_demo_data_ob,ordi_behav_vars ,cluster_col='clusterID')
    behav_results.append(behav_df)

In [ ]:
filtered_dfs=[df[df['Significant_FDR'] == True] for df in behav_results]
filtered_dfs=pd.concat(filtered_dfs, ignore_index=True)
filtered_dfs

In [ ]:
behav_results[5]

In [ ]:
import pickle
with open(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/BehavTest/{band}_behav_q.pkl", "wb") as f:
    pickle.dump(behav_results, f)

    # Load it back later
#with open(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/{band}_behav_q.pkl", "rb") as f:
 #   loaded_list = pickle.load(f)

In [ ]:
import scikit_posthocs as sp_post
from scipy.stats import kruskal, mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns

def create_boxplot_with_significance(var_ofint, noal_sorted_demo_data_ob):
    ddd = noal_sorted_demo_data_ob[[var_ofint, 'clusterID']]
    
    # Statistical tests - choose based on number of clusters
    clusters = sorted(ddd['clusterID'].unique())
    n_clusters = len(clusters)
    
    print(f"Number of clusters: {n_clusters}")
    
    if n_clusters == 2:
        # Use Mann-Whitney U test for 2 groups
        group1 = ddd[ddd['clusterID'] == clusters[0]][var_ofint]
        group2 = ddd[ddd['clusterID'] == clusters[1]][var_ofint]
        mw_stat, mw_p = mannwhitneyu(group1, group2, alternative='two-sided')
        print(f"Mann-Whitney U={mw_stat:.3f}, p={mw_p:.3f}")
        
        # For 2 groups, use the original p-value for consistency
        # but still show the Dunn's result for completeness
        dunn_results = sp_post.posthoc_dunn(ddd, val_col=var_ofint, group_col='clusterID', p_adjust='fdr_bh')
        print("\nDunn's Test (FDR corrected) for reference:")
        print(dunn_results)
        
        # Use the original Mann-Whitney p-value for significance determination
        significant_pairs = []
        if mw_p < 0.05:
            significant_pairs.append((clusters[0], clusters[1], mw_p))
    
    elif n_clusters > 2:
        # Use Kruskal-Wallis with Dunn's post-hoc for 3+ groups
        groups = [ddd[ddd['clusterID'] == g][var_ofint] for g in clusters]
        kw_stat, kw_p = kruskal(*groups)
        print(f"Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.3f}")
        
        # Dunn's post-hoc test
        dunn_results = sp_post.posthoc_dunn(ddd, val_col=var_ofint, group_col='clusterID', p_adjust='fdr_bh')
        print("\nDunn's Test Pairwise Comparisons (FDR corrected):")
        print(dunn_results)
        
        # Find significant pairs using Dunn's corrected p-values
        significant_pairs = []
        for i in range(n_clusters):
            for j in range(i+1, n_clusters):
                p_val = dunn_results.iloc[i, j]
                if p_val < 0.05:
                    significant_pairs.append((clusters[i], clusters[j], p_val))
    
    else:
        print("Only one cluster found - no statistical comparisons possible")
        significant_pairs = []
        dunn_results = None
    
    # Create figure (rest of your plotting code remains the same)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=ddd, x='clusterID', y=var_ofint, ax=ax)
    ax.set_xlabel('Cluster ID')
    ax.set_ylabel(var_ofint)
    
    # Define significance levels
    def get_significance_stars(p_value):
        if p_value < 0.001:
            return '***'
        elif p_value < 0.01:
            return '**'
        elif p_value < 0.05:
            return '*'
        else:
            return 'ns'
    
    def add_significance_bar(ax, x1, x2, y, text):
        ax.plot([x1, x1, x2, x2], [y, y+0.05, y+0.05, y], lw=1.5, color='black')
        ax.text((x1+x2)*0.5, y+0.03, text, ha='center', va='bottom', color='black')
    
    # Add significance bars
    if significant_pairs:
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        significant_pairs.sort(key=lambda x: x[2])
        
        for idx, (cluster1, cluster2, p_val) in enumerate(significant_pairs):
            x1 = clusters.index(cluster1)
            x2 = clusters.index(cluster2)
            y_pos = y_max - (idx + 1) * 0.08 * y_range
            stars = get_significance_stars(p_val)
            add_significance_bar(ax, x1, x2, y_pos, stars)
            
            if y_pos > y_max:
                ax.set_ylim(y_min, y_pos + 0.1 * y_range)
    
    plt.tight_layout()
    
    # Print summary with consistent p-values
    if n_clusters >= 2:
        if n_clusters == 2:
            print(f"\nSignificant pairwise comparisons (p < 0.05) - Mann-Whitney test:")
            if significant_pairs:
                for cluster1, cluster2, p_val in significant_pairs:
                    stars = get_significance_stars(p_val)
                    print(f"Cluster {cluster1} vs Cluster {cluster2}: p = {p_val:.4f} {stars}")
            else:
                print("No significant pairwise comparisons found.")
        else:
            print(f"\nSignificant pairwise comparisons (p < 0.05) - Dunn's test:")
            if significant_pairs:
                for cluster1, cluster2, p_val in significant_pairs:
                    stars = get_significance_stars(p_val)
                    print(f"Cluster {cluster1} vs Cluster {cluster2}: p = {p_val:.4f} {stars}")
            else:
                print("No significant pairwise comparisons found.")
    
    return fig

In [ ]:
noal_sorted_demo_data_ob[filtered_dfs['Variable'].tolist() + ['clusterID']].groupby("clusterID").mean(numeric_only=True)

In [ ]:
marginal_vars=['ADD_EX','ADD_MUS','ADD_INT']
noal_sorted_demo_data_ob[marginal_vars + ['clusterID']].groupby("clusterID").mean(numeric_only=True)

In [ ]:
var_ofint=['BL_FULL45'	,'BL_HOWMUCH0',	'BL_HOWMUCH45','ADD_EX','ADD_MUS','ADD_INT']
for i in var_ofint:
    fig = create_boxplot_with_significance(i,noal_sorted_demo_data_ob)
    fig.savefig(f'/home/jupy/Subtypes_Obesity_Clustering/Outputs/Anthron_Test/{band}_boxplot_{i}.png', dpi=300, bbox_inches='tight')

In [ ]:
from cliffs_delta import cliffs_delta  
# Get unique clusters
cluster_ids = sorted(noal_sorted_demo_data_ob['clusterID'].unique())

for var in var_ofint:
    # Only proceed if exactly 2 clusters; otherwise compute pairwise
    if len(cluster_ids) == 2:
        group1 = noal_sorted_demo_data_ob[noal_sorted_demo_data_ob['clusterID'] == cluster_ids[0]][var].dropna()
        group2 = noal_sorted_demo_data_ob[noal_sorted_demo_data_ob['clusterID'] == cluster_ids[1]][var].dropna()
        
        delta, res = cliffs_delta(group1, group2)
        print(f"{var}: Cliff's delta = {delta:.3f}, magnitude = {res}")
    else:
        print(f"{var}: More than 2 groups – consider pairwise comparisons")

#### Compare ROI WDFC between clusters

In [ ]:
shapdf=shap_results['component_1']['feature_importance']
important_roi1=np.array(shapdf[shapdf['mean_abs_shap']>0.1]['feature'].astype(str))

shapdf=shap_results['component_0']['feature_importance']
important_roi0=np.array(shapdf[shapdf['mean_abs_shap']>0.1]['feature'].astype(str))

important_roi=important_roi1.tolist()+important_roi0.tolist()
important_roi=list(set(important_roi))

In [ ]:
fc_ob_cluster0=adjusted_fc_ob[(cluster_results_obese['global_clusters']==0) ]
fc_ob_cluster0=fc_ob_cluster0[important_roi]

fc_ob_cluster1=adjusted_fc_ob[(cluster_results_obese['global_clusters']==1)]
fc_ob_cluster1=fc_ob_cluster1[important_roi]


In [ ]:
from scipy.stats import shapiro, mannwhitneyu
import statsmodels.stats.multitest as multi

def compare_clusters_features(*dfs, alpha=0.05):
    """
    Compare each feature across multiple clusters with proper statistical testing
    
    Parameters:
    *dfs: variable number of dataframes (each representing a cluster)
    alpha: significance level
    
    Returns:
    DataFrame with comparison results
    """
    results = []
    n_clusters = len(dfs)
    
    # Get all unique feature names across all dataframes
    all_features = set()
    for df in dfs:
        all_features.update(df.columns)
    all_features = sorted(list(all_features))
    
    for feature in all_features:
        # Extract data for this feature from all clusters
        cluster_data = []
        valid_clusters = []
        
        for i, df in enumerate(dfs):
            if feature in df.columns:
                data = df[feature].dropna()
                if len(data) > 0:  # Only include clusters with data for this feature
                    cluster_data.append(data)
                    valid_clusters.append(i)
        
        if len(cluster_data) < 2:
            # Skip if less than 2 clusters have data for this feature
            continue
        
        # Check normality for all groups
        normality_results = []
        for data in cluster_data:
            if len(data) >= 3:  # Shapiro-Wilk requires at least 3 observations
                _, p_norm = shapiro(data)
                normality_results.append(p_norm > alpha)
            else:
                normality_results.append(False)  # Assume non-normal for small samples
        
        all_normal = all(normality_results)
        
        # Choose appropriate test
        if all_normal and len(cluster_data) == 2:
            # Use t-test for 2 groups with normal data
            stat, p_val = stats.ttest_ind(cluster_data[0], cluster_data[1])
            test_used = "t-test"
        elif len(cluster_data) == 2:
            # Use Mann-Whitney U test for 2 groups with non-normal data
            stat, p_val = mannwhitneyu(cluster_data[0], cluster_data[1], alternative='two-sided')
            test_used = "Mann-Whitney U"
        else:
            # Use Kruskal-Wallis test for 3+ groups
            stat, p_val = stats.kruskal(*cluster_data)
            test_used = "Kruskal-Wallis"
        
        # Calculate descriptive statistics for each cluster
        cluster_stats = {}
        for i, data in enumerate(cluster_data):
            cluster_stats[f'cluster{valid_clusters[i]}_mean'] = np.mean(data)
            cluster_stats[f'cluster{valid_clusters[i]}_std'] = np.std(data)
            cluster_stats[f'cluster{valid_clusters[i]}_n'] = len(data)
            cluster_stats[f'cluster{valid_clusters[i]}_normal'] = normality_results[i]
        
        # Calculate effect size (Cohen's d for 2 groups, eta-squared for 3+ groups)
        if len(cluster_data) == 2:
            # Cohen's d for 2 groups
            n1, n2 = len(cluster_data[0]), len(cluster_data[1])
            pooled_std = np.sqrt(((n1-1)*np.var(cluster_data[0], ddof=1) + (n2-1)*np.var(cluster_data[1], ddof=1)) / (n1+n2-2))
            effect_size = (np.mean(cluster_data[0]) - np.mean(cluster_data[1])) / pooled_std
            effect_size_name = "cohens_d"
        else:
            # Eta-squared for 3+ groups (simplified calculation)
            grand_mean = np.mean(np.concatenate(cluster_data))
            ss_between = sum(len(data) * (np.mean(data) - grand_mean)**2 for data in cluster_data)
            ss_total = sum(np.sum((data - grand_mean)**2) for data in cluster_data)
            effect_size = ss_between / ss_total if ss_total > 0 else 0
            effect_size_name = "eta_squared"
        
        result = {
            'feature': feature,
            'statistic': stat,
            'p_value': p_val,
            'test_used': test_used,
            'n_clusters': len(cluster_data),
            'clusters_compared': valid_clusters,
            'significant': p_val < alpha,
            effect_size_name: effect_size
        }
        result.update(cluster_stats)
        results.append(result)
    
    return pd.DataFrame(results)


In [ ]:
DWFC_compare_results=compare_clusters_features(fc_ob_cluster0, fc_ob_cluster1) #  ,fc_ob_cluster2,fc_ob_cluster3

In [ ]:
# Apply FDR correction for multiple comparisons
rejected, pvals_corrected, _, _ = multi.multipletests(
    DWFC_compare_results['p_value'], alpha=0.05, method='fdr_bh'
)

DWFC_compare_results['p_value_fdr'] = pvals_corrected
DWFC_compare_results['significant_fdr'] = rejected

print("Statistical Comparison Results:")
print("=" * 80)
print(f"Total features: {len(DWFC_compare_results)}")
print(f"Significant before FDR: {DWFC_compare_results['significant'].sum()}")
print(f"Significant after FDR: {DWFC_compare_results['significant_fdr'].sum()}")
print(f"Number of clusters compared: {DWFC_compare_results['n_clusters'].iloc[0]}")

# For post-hoc testing when Kruskal-Wallis is significant and there are 3+ clusters
if DWFC_compare_results['n_clusters'].iloc[0] > 2:
    print("\nNote: For significant results with 3+ clusters, consider post-hoc pairwise comparisons.")
    
    # Example of how to do post-hoc testing for significant features
    significant_features = DWFC_compare_results[DWFC_compare_results['significant_fdr']]['feature']
    print(f"Features with significant differences: {len(significant_features)}")

#### Read Brain Structure and BrodMann

In [ ]:
ROI_df=pd.read_csv("/home/jupy/88ROIallBA-ROI-ROI.csv",header=None)
ROI_df.columns=['X','Y', 'Z', 'Lobe','Structure','BA','ROI']
ROI_df.head(2)

In [ ]:
DWFC_compare_results["roi_num"] = DWFC_compare_results["feature"].str.replace("roi", "", regex=False).astype(int)
merged_roi_df = ROI_df.merge(DWFC_compare_results, right_on="roi_num", left_on="ROI", how="inner")
merged_roi_df.head(2)

DWFC_compare_results_BA_added=DWFC_compare_results.merge(merged_roi_df[["BA", 
                                                "ROI"]].drop_duplicates(), right_on="ROI", left_on="roi_num", how="inner").drop(columns=['ROI'])
DWFC_compare_results_BA_added["BA"]=DWFC_compare_results_BA_added["BA"].str.replace("rodmann area ", "A ", regex=False).astype(str)
DWFC_compare_results_BA_added=DWFC_compare_results_BA_added.sort_values('BA')

DWFC_compare_results_BA_added = DWFC_compare_results_BA_added.sort_values('BA', 
    key=lambda x: x.str.extract('(\d+)', expand=False).astype(int) )

In [ ]:
def create_grouped_barplot(results_df, x_axis_var, top_n=20, figsize=(15, 8), 
                          cluster_names=None, band=None):
    
    # Determine number of clusters from the results
    n_clusters = results_df['n_clusters'].iloc[0] if 'n_clusters' in results_df.columns else 2
    
    # Sort by effect size and select top N features
    if n_clusters == 2:
        effect_size_col = 'cohens_d'
    else:
        effect_size_col = 'eta_squared'
    
    plot_df = results_df.nlargest(top_n, effect_size_col, keep='all')
    
    # Create a copy and sort by BA number if available
    plot_df = plot_df.copy()
    if 'BA' in plot_df.columns:
        plot_df['BA_sort'] = plot_df['BA'].str.extract('(\d+)').astype(int)
        plot_df = plot_df.sort_values('BA_sort')
    
    features = plot_df[x_axis_var].values
    p_values = plot_df['p_value_fdr'].values
    
    # Set up the plot
    x = np.arange(len(features))
    
    if n_clusters == 2:
        # Original two-cluster plotting logic
        width = 0.35
        
        means0 = plot_df['cluster0_mean'].values
        means1 = plot_df['cluster1_mean'].values
        std0 = plot_df['cluster0_std'].values
        std1 = plot_df['cluster1_std'].values
        
        fig, ax = plt.subplots(figsize=figsize)
        
        # Create bars for two clusters
        bar1 = ax.bar(x - width/2, means0, width, 
                     label=cluster_names[0] if cluster_names else 'Cluster 0',
                     color='red', alpha=0.7, yerr=std0, capsize=3)
        bar2 = ax.bar(x + width/2, means1, width, 
                     label=cluster_names[1] if cluster_names else 'Cluster 1',
                     color='blue', alpha=0.7, yerr=std1, capsize=3)
        
        # Add significance stars
        for i, p_val in enumerate(p_values):
            if p_val < 0.001:
                star = '***'
            elif p_val < 0.01:
                star = '**'
            elif p_val < 0.05:
                star = '*'
            else:
                star = ''
            
            if star:
                y_pos = max(means0[i], means1[i]) + max(std0[i], std1[i]) - 0.1
                ax.text(x[i], y_pos, star, ha='center', va='bottom', 
                       fontweight='bold', fontsize=12)
        
    else:
        # Multi-cluster plotting (3+ clusters)
        width = 0.8 / n_clusters
        colors = plt.cm.Set3(np.linspace(0, 1, n_clusters))
        
        fig, ax = plt.subplots(figsize=(max(15, len(features)*0.8), 8))
        
        # Extract means and stds for all clusters
        bars = []
        for cluster_idx in range(n_clusters):
            means = plot_df[f'cluster{cluster_idx}_mean'].values
            stds = plot_df[f'cluster{cluster_idx}_std'].values
            
            bar = ax.bar(x + (cluster_idx - n_clusters/2 + 0.5) * width, 
                        means, width,
                        label=cluster_names[cluster_idx] if cluster_names else f'Cluster {cluster_idx}',
                        color=colors[cluster_idx], alpha=0.7, yerr=stds, capsize=3)
            bars.append(bar)
        
        # For multi-cluster, show overall significance from Kruskal-Wallis
        # and indicate which features have significant overall differences
        for i, p_val in enumerate(p_values):
            if p_val < 0.05:
                # Add significance marker above the bars
                y_max = max([plot_df[f'cluster{j}_mean'].iloc[i] + plot_df[f'cluster{j}_std'].iloc[i] 
                           for j in range(n_clusters)])
                ax.text(x[i], y_max + 0.15, '*', ha='center', va='bottom', 
                       fontweight='bold', fontsize=14, color='red')
    
    # Create x-axis labels
    x_labels = []
    for idx, row in plot_df.iterrows():
        if 'BA' in plot_df.columns and 'roi_num' in plot_df.columns:
            ba_num = row['BA'] if pd.notna(row['BA']) else 'N/A'
            roi_num = row['roi_num'] if pd.notna(row['roi_num']) else 'N/A'
            x_labels.append(f'{ba_num}\nROI {roi_num}')
        else:
            # Use the feature name directly
            feature_name = str(row[x_axis_var])
            if len(feature_name) > 15:
                feature_name = feature_name[:12] + '...'
            x_labels.append(feature_name)
    
    # Customize the plot
    ax.set_ylabel('Weighted Degree Connectivity', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig, ax

print('Bars show mean ± SD, *p<0.05, **p<0.01, ***p<0.001, FDR-corrected')

In [ ]:
fig, ax = create_grouped_barplot(DWFC_compare_results_BA_added, 'BA', 
                                  cluster_names=['Cluster 0', 'Cluster 1']) #  ,'Cluster 2','Cluster 3'
fig.savefig(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/FC_figures/{band}_DWFC_barplot.png", dpi=300, bbox_inches="tight")
fig.savefig(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/FC_figures/{band}_DWFC_barplot.pdf", bbox_inches="tight")

In [ ]:
#DWFC_compare_results_BA_added.to_csv(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/Others_for_Manuscript/DWFC_test_{band}.csv", index=False)

In [ ]:
DWFC_compare_results_BA_added

In [ ]:
import scikit_posthocs as sp
from scipy.stats import mannwhitneyu

fc_ob_cluster0['ClusterID'] = 0
fc_ob_cluster1['ClusterID'] = 1

fc_ob_clusters = [fc_ob_cluster0, fc_ob_cluster1]

# Add ClusterID column dynamically
for idx, df in enumerate(fc_ob_clusters):
    df['ClusterID'] = idx

# Concatenate all clusters
df_fcall = pd.concat(fc_ob_clusters, ignore_index=True)

# Write directly to file
with open(f"/home/jupy/Subtypes_Obesity_Clustering/Outputs/FC_kruskal_dunn/{band}_KD.txt", "w") as f:
    # Loop over features
    for col in fc_ob_clusters[0].columns.tolist():
        if col != 'ClusterID':
            f.write(f"Feature: {col}\n")
            
            # Mann-Whitney U test (for 2 groups)
            group0 = fc_ob_clusters[0][col]
            group1 = fc_ob_clusters[1][col]
            stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
            
            # Calculate sample sizes
            n0 = len(group0)
            n1 = len(group1)
            n_total = n0 + n1
            
            # Rank-biserial correlation (effect size for Mann-Whitney)
            # r = 1 - (2*U) / (n0*n1)
            rank_biserial = 1 - (2 * stat) / (n0 * n1)
            
            f.write(f"Mann-Whitney U = {stat:.4f}, p = {p:.4e}\n")
            f.write(f"Sample sizes: Cluster 0 (n={n0}), Cluster 1 (n={n1})\n")
            f.write(f"Rank-biserial correlation (effect size) = {rank_biserial:.4f}\n")
            
            # For 2 groups, Dunn's test is equivalent to Mann-Whitney with correction
            # but we'll include it for consistency with your original format
            dunn_fdr = sp.posthoc_dunn(df_fcall, val_col=col, group_col="ClusterID", p_adjust="fdr_bh")
            f.write("Dunn's test results (FDR-adjusted p-values):\n")
            f.write(f"{dunn_fdr}\n")
            
            # Also add interpretation of effect size
            if abs(rank_biserial) < 0.3:
                effect_size_desc = "small"
            elif abs(rank_biserial) < 0.5:
                effect_size_desc = "medium"
            else:
                effect_size_desc = "large"
            
            f.write(f"Effect size interpretation: {effect_size_desc}\n")
            f.write('\n')
            
            # Also print to console if you want to see progress
            print(f"Processed: {col}")

print("Analysis complete! Results saved to file.")

# ------------------------------------------ END -------------------------------------------